In [8]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.datasets import fetch_20newsgroups
from collections import Counter
from scipy.sparse import dok_matrix, csr_matrix
from scipy.sparse.linalg import svds

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [9]:
# Load the 20 Newsgroups dataset
newsgroups = fetch_20newsgroups(subset='all')
documents = newsgroups.data

# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation
    words = text.split()
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Preprocess each document
processed_docs = [preprocess_text(doc) for doc in documents]


In [16]:
processed_docs

['mamatha devineni ratnam mr47 andrew cmu edu subject pen fan reaction organization post office carnegie mellon pittsburgh pa line 12 nntp posting host po4 andrew cmu edu sure bashers pen fan pretty confused lack kind post recent pen massacre devil actually bit puzzled bit relieved however going put end non pittsburghers relief bit praise pen man killing devil worse thought jagr showed much better regular season stats also lot fo fun watch playoff bowman let jagr lot fun next couple game since pen going beat pulp jersey anyway disappointed see islander lose final regular season game pen rule',
 'mblawson midway ecn uoknor edu matthew b lawson subject high performance vlb video card summary seek recommendation vlb video card nntp posting host midway ecn uoknor edu organization engineering computer network university oklahoma norman ok usa keywords orchid stealth vlb line 21 brother market high performance video card support vesa local bus 1 2mb ram anyone suggestion idea diamond stealth

In [17]:
# Create vocabulary and document-term matrix as a sparse matrix
def create_vocab(docs):
    vocab = Counter()
    for doc in docs:
        for word in doc.split():
            vocab[word] += 1
    return sorted(vocab.keys())

# Create vocabulary and sparse document-term matrix
vocab = create_vocab(processed_docs)
vocab_index = {word: i for i, word in enumerate(vocab)}

# Initialize a sparse document-term matrix in DOK format
doc_term_matrix_sparse = dok_matrix((len(processed_docs), len(vocab)), dtype=int)

# Fill in the sparse document-term matrix
for doc_idx, doc in enumerate(processed_docs):
    for word in doc.split():
        if word in vocab_index:
            term_idx = vocab_index[word]
            doc_term_matrix_sparse[doc_idx, term_idx] += 1

# Convert to CSR format for efficient SVD
doc_term_matrix_sparse = doc_term_matrix_sparse.tocsr()


In [21]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

# Convert the document-term matrix to float type
doc_term_matrix_sparse = doc_term_matrix_sparse.astype(float)

# Function to apply SVD
def apply_svd(sparse_matrix, num_components):
    # Perform SVD on the sparse matrix
    U, Sigma, VT = svds(sparse_matrix, k=num_components)
    Sigma = np.diag(Sigma)
    return U, Sigma, VT

# Experiment with different numbers of components
num_components_list = [3, 4, 5]
svd_results = {k: apply_svd(doc_term_matrix_sparse, k) for k in num_components_list}


In [23]:
def calculate_probabilities(U, Sigma, VT):
    # Topic-Word Probability Matrix
    topic_word_matrix = VT.T
    topic_word_prob = topic_word_matrix / topic_word_matrix.sum(axis=0, keepdims=True)

    # Document-Topic Probability Matrix
    document_topic_matrix = U @ Sigma
    document_topic_prob = document_topic_matrix / document_topic_matrix.sum(axis=1, keepdims=True)
    
    return topic_word_prob, document_topic_prob

# Calculate probabilities for each number of components
probabilities = {k: calculate_probabilities(*svd_results[k]) for k in num_components_list}


In [25]:
# Display top words and sample document-topic probabilities for each number of components
for k, (topic_word_prob, document_topic_prob) in probabilities.items():
    print(f"\nNumber of Components (Topics): {k}")
    
    # Display the top words per topic
    for i in range(k):
        top_word_indices = topic_word_prob[:, i].argsort()[::-1][:10]
        top_words = [vocab[idx] for idx in top_word_indices]
        print(f"Top words for Topic {i + 1}: {', '.join(top_words)}")
    
    # Display sample document-topic probabilities
    print("\nSample Document-Topic Probabilities:")
    sample_docs = document_topic_prob[:5]  # Display for first 5 documents
    for doc_idx, doc_probs in enumerate(sample_docs):
        topic_probs = ", ".join([f"Topic {i+1}: {prob:.2f}" for i, prob in enumerate(doc_probs)])
        print(f"Document {doc_idx + 1}: {topic_probs}")



Number of Components (Topics): 3
Top words for Topic 1: x, file, entry, output, program, oname, printf, line, char, stream
Top words for Topic 2: 0, w, 1, x, r, 2, q, p, g, 3
Top words for Topic 3: ax, max, q, 3, p, r, 7, g, g9v, n

Sample Document-Topic Probabilities:
Document 1: Topic 1: 2.67, Topic 2: -1.65, Topic 3: -0.02
Document 2: Topic 1: -0.04, Topic 2: 1.00, Topic 3: 0.04
Document 3: Topic 1: -7.16, Topic 2: 7.97, Topic 3: 0.19
Document 4: Topic 1: -0.11, Topic 2: 1.05, Topic 3: 0.06
Document 5: Topic 1: 0.07, Topic 2: 0.90, Topic 3: 0.04

Number of Components (Topics): 4
Top words for Topic 1: 0, 1, 2, 4, 10, 16, 12, 3, 14, 15
Top words for Topic 2: x, file, entry, output, program, oname, printf, line, char, stream
Top words for Topic 3: 0, w, 1, x, r, 2, q, p, g, 3
Top words for Topic 4: ax, max, q, 3, p, r, 7, g, g9v, n

Sample Document-Topic Probabilities:
Document 1: Topic 1: -9.02, Topic 2: 6.22, Topic 3: 3.85, Topic 4: -0.05
Document 2: Topic 1: -0.32, Topic 2: 0.05, 